In [2]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

PARQUET_PATH = Path("../data/processed/pgfn/pgfn_sida.parquet")
pq_file = pq.ParquetFile(PARQUET_PATH)

print("Linhas:", f"{pq_file.metadata.num_rows:,}")
print("Colunas:", len(pq_file.schema_arrow.names))
print("Row groups:", pq_file.num_row_groups)

rg0 = pq_file.read_row_group(0).to_pandas()
head_df = rg0.head(20)

pd.set_option("display.max_columns", 200)
display(head_df)


Linhas: 536,462,590
Colunas: 18
Row groups: 2888


,cpf_cnpj,tipo_pessoa,tipo_devedor,nome_devedor,uf_unidade_responsavel,unidade_responsavel,numero_inscricao,tipo_situacao_inscricao,situacao_inscricao,receita_principal,data_inscricao,indicador_ajuizado,valor_consolidado,uf_devedor,ano,trimestre,uf,arquivo_origem
0,XXX868.239XX,Pessoa física,CORRESPONSAVEL,JOCELI ANTONIO RIGO,AC,ACRE,2229700000188,Em cobrança,ATIVA AJUIZADA,Receita da dívida ativa - IRPJ,16/01/1997,SIM,1879466.0,None,2020,1,AC,arquivo_lai_SIDA_AC_202002.csv
1,XXX640.502XX,Pessoa física,PRINCIPAL,HONORINA FERNANDES MOSLE,AC,ACRE,2261200040184,Em cobrança,ATIVA AJUIZADA,R D Ativa - Multa Isolada,10/10/2012,SIM,843112.0,None,2020,1,AC,arquivo_lai_SIDA_AC_202002.csv
2,07.075.587/0001-47,Pessoa jurídica,PRINCIPAL,A. L. GONCALVES,AC,ACRE,2241700084607,Em cobrança,ATIVA NAO PRIORIZADA PARA AJUIZAMENTO,DIV ATIVA-SIMPLES NACIONAL,14/06/2017,NAO,511167.0,None,2020,1,AC,arquivo_lai_SIDA_AC_202002.csv
3,XXX520.192XX,Pessoa física,CORRESPONSAVEL,ALCIVANIO FERREIRA DA SILVA,AC,ACRE,2241700079794,Em cobrança,ATIVA NAO PRIORIZADA PARA AJUIZAMENTO,DIV ATIVA-SIMPLES NACIONAL,14/06/2017,NAO,215373.0,None,2020,1,AC,arquivo_lai_SIDA_AC_202002.csv
4,XXX152.082XX,Pessoa física,CORRESPONSAVEL,FABRICIA ARAUJO DA ROCHA,AC,ACRE,2261500098080,Em cobrança,ATIVA AJUIZADA,Receita da dívida ativa - COFINS,25/08/2015,SIM,5873119.0,None,2020,1,AC,arquivo_lai_SIDA_AC_202002.csv
5,04.092.011/0001-08,Pessoa jurídica,PRINCIPAL,MARIO JOSE FURTADO,AC,ACRE,2251100041068,Em cobrança,ATIVA NAO PRIORIZADA PARA AJUIZAMENTO,Receita da dívida ativa - Multa - CLT,03/10/2011,NAO,304549.0,None,2020,1,AC,arquivo_lai_SIDA_AC_202002.csv
6,01.634.845/0001-00,Pessoa jurídica,PRINCIPAL,SERVICO DE AGUA E ESGOTO DE RIO BRANCO,AC,ACRE,2251600078777,Benefício Fiscal,ATIVA NAO AJUIZAVEL NEGOCIADA NO SISPAR,Receita da dívida ativa - Multa - CLT,07/10/2016,NAO,540894.0,None,2020,1,AC,arquivo_lai_SIDA_AC_202002.csv
7,XXX393.292XX,Pessoa física,CORRESPONSAVEL,BRUNO COTTA PAIVA,AC,ACRE,2251700046680,Em cobrança,ATIVA A SER AJUIZADA,Receita da dívida ativa - Multa - CLT,04/05/2017,NAO,145802.0,None,2020,1,AC,arquivo_lai_SIDA_AC_202002.csv
8,XXX393.292XX,Pessoa física,CORRESPONSAVEL,BRUNO COTTA PAIVA,AC,ACRE,2241600083609,Em cobrança,ATIVA AJUIZADA,Receita da dívida ativa - SIMPLES,16/09/2016,SIM,3328962.0,None,2020,1,AC,arquivo_lai_SIDA_AC_202002.csv
9,XXX150.042XX,Pessoa física,CORRESPONSAVEL,CARLOS ANDRE DA COSTA NERY SILVA,AC,ACRE,2241900057506,Em cobrança,ATIVA EM COBRANCA,DIV ATIVA-SIMPLES NACIONAL,18/06/2019,NAO,131167.0,None,2020,1,AC,arquivo_lai_SIDA_AC_202002.csv


In [3]:
MAX_SAMPLE_ROWS = 200_000

parts = [rg0]
count = len(rg0)
rg = 1

while count < MAX_SAMPLE_ROWS and rg < pq_file.num_row_groups:
    part = pq_file.read_row_group(rg).to_pandas()
    parts.append(part)
    count += len(part)
    rg += 1

sample_df = pd.concat(parts, ignore_index=True).head(MAX_SAMPLE_ROWS)
sample_df.shape


(200000, 18)

In [4]:
na_abs = sample_df.isna().sum()
na_pct = (na_abs / len(sample_df) * 100).round(2)
na_tbl = pd.DataFrame({"nulos": na_abs, "pct": na_pct}).sort_values("nulos", ascending=False)
na_tbl.head(20)


,nulos,pct
uf_devedor,200000,100.0
cpf_cnpj,0,0.0
tipo_pessoa,0,0.0
tipo_devedor,0,0.0
uf_unidade_responsavel,0,0.0
nome_devedor,0,0.0
numero_inscricao,0,0.0
tipo_situacao_inscricao,0,0.0
situacao_inscricao,0,0.0
unidade_responsavel,0,0.0


In [5]:
for c in ["uf", "tipo_pessoa", "tipo_devedor", "tipo_situacao_inscricao", "situacao_inscricao", "indicador_ajuizado", "ano", "trimestre"]:
    if c in sample_df.columns:
        display(sample_df[c].value_counts(dropna=False).head(15))


uf
AL    151994
AC     48006
Name: count, dtype: int64

tipo_pessoa
Pessoa jurídica    102918
Pessoa física       97082
Name: count, dtype: int64

tipo_devedor
PRINCIPAL         151863
CORRESPONSAVEL     48038
SOLIDARIO             99
Name: count, dtype: int64

tipo_situacao_inscricao
Em cobrança                      172882
Benefício Fiscal                  26234
Garantia                            421
Suspenso por decisão judicial       338
Em negociação                       125
Name: count, dtype: int64

situacao_inscricao
ATIVA AJUIZADA                                                           64078
ATIVA NAO PRIORIZADA PARA AJUIZAMENTO                                    53277
ATIVA EM COBRANCA                                                        31685
ATIVA NAO AJUIZAVEL NEGOCIADA NO SISPAR                                  14897
ATIVA A SER AJUIZADA                                                      9501
ATIVA AJUIZADA NEGOCIADA NO SISPAR                                        8375
ATIVA COM AJUIZAMENTO A SER PROSSEGUIDO                                   5229
ATIVA A SER COBRADA                                                       4900
ATIVA COM PARCELAMENTO SIMPLIFICADO RESCINDIDO E AJUIZAM A PROSSEGUIR     2069
ATIVA AJUIZADA PARCELADA LEI 12996/14                                      795
ATIVA AJUIZADA PARC LEI 11941/09 ART 3-SALDO REMANESCENTE PARCEL           742
ATIVA COM PARCELAMENTO RESCINDIDO E AJUIZAMENTO A SER PROSSEGUIDO          655
ATIVA NAO AJUIZAVEL EM PROCESSO D

indicador_ajuizado
NAO    116075
SIM     83925
Name: count, dtype: int64

ano
2020    200000
Name: count, dtype: int64

trimestre
1    200000
Name: count, dtype: int64